In [95]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
import seaborn as sns
import sqlite3
import os
from scipy import stats
from statsmodels.stats.multitest import multipletests
from pprint import pprint

In [44]:
con = sqlite3.connect("../data/bdb2027.sqlite")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))  # list tables
# df = pd.read_sql("SELECT * FROM <table_name>", con)

                          name
0      player_career_successes
1                  player_play
2           game_tracking_2023
3                      players
4             combine_tracking
5           game_tracking_2024
6                        games
7           game_tracking_2025
8              combine_results
9                   df_players
10         df_combine_tracking
11                    df_games
12  df_player_career_successes
13          df_combine_results
14           game_tracking_all
15    df_tracking_summary_year
16             df_play_summary


# Part 1: Data Engineering & Prep

In [45]:
def parse_tracking_time(series: pd.Series) -> pd.Series:
    """
    Handles SQLite TEXT timestamps (mixed ISO/fractional strings) or numeric
    relative seconds stored as strings/numbers.
    """
    if pd.api.types.is_datetime64_any_dtype(series) or pd.api.types.is_timedelta64_dtype(series):
        return series

    cleaned = series.replace(
        {"": pd.NA, "None": pd.NA, "none": pd.NA, "NA": pd.NA, "null": pd.NA, "NULL": pd.NA}
    )
    dt = pd.to_datetime(cleaned, format="mixed", errors="coerce")
    if dt.notna().any():
        return dt

    numeric = pd.to_numeric(cleaned, errors="coerce")
    return pd.to_timedelta(numeric, unit="s")


def compute_elapsed_seconds(time_series: pd.Series) -> np.ndarray:
    """Converts parsed timestamps or timedeltas into elapsed seconds from t[0]."""
    parsed = parse_tracking_time(time_series)
    if pd.api.types.is_datetime64_any_dtype(parsed) or pd.api.types.is_timedelta64_dtype(parsed):
        t_rel = (parsed - parsed.iloc[0]).dt.total_seconds().to_numpy(dtype=float)
    else:
        t_rel = (parsed.to_numpy(dtype=float) - float(parsed.iloc[0]))

    # Forward/back-fill any isolated NaT/NaN timestamps so 10 Hz spacing holds
    s = pd.Series(t_rel).interpolate(method="linear", limit_direction="both")
    vals = s.to_numpy(dtype=float)
    if np.isnan(vals).all():
        return np.arange(len(vals), dtype=float) * 0.1
    return vals


def unwrap_heading_deg(df: pd.DataFrame) -> np.ndarray:
    """Unwraps `dir` (preferred), then `o`, then `(x, y)` trajectory heading in degrees."""
    for col in ("dir", "o"):
        if col in df.columns and df[col].notna().any():
            vals = df[col].interpolate(method="linear", limit_direction="both").to_numpy(dtype=float)
            return np.rad2deg(np.unwrap(np.deg2rad(vals)))

    if {"x", "y"}.issubset(df.columns):
        x = df["x"].interpolate(method="linear", limit_direction="both").to_numpy(dtype=float)
        y = df["y"].interpolate(method="linear", limit_direction="both").to_numpy(dtype=float)
        dx = np.diff(x, prepend=x[0])
        dy = np.diff(y, prepend=y[0])
        return np.rad2deg(np.unwrap(np.arctan2(dy, dx)))

    return np.zeros(len(df), dtype=float)


def compute_segment_kinematics(seg_df: pd.DataFrame) -> dict[str, float]:
    """
    Computes standardized kinematic metrics on any ordered tracking slice
    (a drill phase, drill attempt, or in-game route window).
    """
    g = seg_df.reset_index(drop=True)
    n_frames = len(g)
    if n_frames == 0:
        return {}

    t_rel = compute_elapsed_seconds(g["time"])
    dt = np.diff(t_rel, prepend=t_rel[0])
    dt = np.where((~np.isfinite(dt)) | (dt <= 0), 0.1, dt)
    duration_s = float(max(0.0, t_rel[-1] - t_rel[0])) if n_frames > 1 else 0.0

    if "dis" in g.columns and g["dis"].notna().any():
        dist_yds = float(g["dis"].fillna(0.0).sum())
    elif {"x", "y"}.issubset(g.columns):
        dx = g["x"].diff().fillna(0.0).to_numpy(dtype=float)
        dy = g["y"].diff().fillna(0.0).to_numpy(dtype=float)
        dist_yds = float(np.hypot(dx, dy).sum())
    else:
        dist_yds = float(np.nansum(g["s"].to_numpy(dtype=float) * dt))

    s_raw = g["s"].interpolate(method="linear", limit_direction="both").to_numpy(dtype=float)
    s_smooth = pd.Series(s_raw).rolling(window=5, center=True, min_periods=1).mean().to_numpy(dtype=float)

    # Longitudinal acceleration (yd/s^2) from smoothed speed
    a_long = np.zeros_like(s_smooth)
    if n_frames >= 2:
        a_long[1:] = np.diff(s_smooth) / dt[1:]
        a_long[0] = a_long[1]

    brake_mags = -a_long[a_long < 0]
    peak_braking = float(np.max(brake_mags)) if len(brake_mags) > 0 else np.nan
    mean_braking = float(np.mean(brake_mags)) if len(brake_mags) > 0 else np.nan
    peak_accel = float(np.nanmax(a_long)) if n_frames > 0 else np.nan

    hdg = unwrap_heading_deg(g)
    turn_steps = np.abs(np.diff(hdg, prepend=hdg[0]))
    turn_rates = np.where(dt > 0, turn_steps / dt, 0.0)

    s_start = float(s_smooth[0])
    s_end = float(s_smooth[-1])
    s_min = float(np.nanmin(s_smooth))
    s_max = float(np.nanmax(s_smooth))

    return {
        "n_frames": int(n_frames),
        "duration_s": duration_s,
        "dist_yds": dist_yds,
        "start_speed_yds": s_start,
        "exit_speed_yds": s_end,
        "min_speed_yds": s_min,
        "peak_speed_yds": s_max,
        "speed_drop_yds": s_max - s_min,
        "speed_gain_yds": s_end - s_start,
        "peak_braking_yds2": peak_braking,
        "mean_braking_yds2": mean_braking,
        "peak_accel_yds2": peak_accel,
        "max_turn_rate_deg_s": float(np.nanmax(turn_rates)) if n_frames > 0 else np.nan,
        "total_turn_deg": float(np.nansum(turn_steps)),
    }

In [46]:
DRILL_METRIC_COLS = [
    "attempt_time_s",
    "duration_s",
    "dist_yds",
    "start_speed_yds",
    "exit_speed_yds",
    "min_speed_yds",
    "peak_speed_yds",
    "speed_drop_yds",
    "speed_gain_yds",
    "peak_braking_yds2",
    "mean_braking_yds2",
    "peak_accel_yds2",
    "max_turn_rate_deg_s",
    "total_turn_deg",
]

def _pick_best_attempts(attempt_summary_df: pd.DataFrame) -> pd.DataFrame:
    """Keeps the single fastest attempt per `(nfl_id, drill)`."""
    if attempt_summary_df.empty:
        return pd.DataFrame(columns=["nfl_id", "drill"])

    df = attempt_summary_df.copy()
    time_col = "attempt_time_s" if df["attempt_time_s"].notna().any() else "duration_s"
    df["_rank_time"] = df[time_col].fillna(np.inf)

    sort_cols = [c for c in ["nfl_id", "drill", "_rank_time", "attempt"] if c in df.columns]
    best = (
        df.sort_values(sort_cols, ascending=[True] * len(sort_cols))
        .drop_duplicates(subset=["nfl_id", "drill"], keep="first")
        .drop(columns=["_rank_time"])
        .reset_index(drop=True)
    )
    return best


def build_drill_kinematics_player(
    combine_tracking_df: pd.DataFrame,
    combine_results_df: pd.DataFrame | None = None,
    drill_phases_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """
    Produces 1 row per `nfl_id` with each player's best-time attempt kinematics
    across drills (`three_cone`, `forty_yd`, `shuttle`).
    """
    if combine_tracking_df.empty:
        return pd.DataFrame(columns=["nfl_id"])

    df = combine_tracking_df.copy()

    # Normalize player ID column name if stored under an alias
    if "nfl_id" not in df.columns:
        for alt in ("player_id", "nflId", "gsis_id"):
            if alt in df.columns:
                df = df.rename(columns={alt: "nfl_id"})
                break
    if "nfl_id" not in df.columns:
        raise KeyError(f"Expected 'nfl_id' in combine_tracking_df, found columns: {list(df.columns)}")

    if "entity_type" in df.columns:
        df = df[df["entity_type"].astype(str).str.upper() == "PLAYER"].copy()

    drill_map = {
        "3-cone": "three_cone",
        "three-cone": "three_cone",
        "three_cone": "three_cone",
        "40-yard": "forty_yd",
        "40_yard": "forty_yd",
        "40-yard dash": "forty_yd",
        "forty": "forty_yd",
        "20-yard shuttle": "shuttle",
        "20_yard_shuttle": "shuttle",
        "short_shuttle": "shuttle",
        "shuttle": "shuttle",
    }
    drill_col = next((c for c in ("drill_type", "drill", "event_name") if c in df.columns), None)
    if drill_col is None:
        raise KeyError(f"No drill identifier column found in combine_tracking_df: {list(df.columns)}")

    df["drill"] = df[drill_col].astype(str).str.lower().str.strip().replace(drill_map)

    group_cols = ["nfl_id", "drill"] + [c for c in ["event_id", "attempt"] if c in df.columns]
    attempt_rows = []
    for keys, g in df.groupby(group_cols, dropna=False, sort=False):
        if len(g) < 5:
            continue
        kin = compute_segment_kinematics(g.sort_values("time"))
        row = dict(zip(group_cols, keys))
        row["attempt_time_s"] = kin["duration_s"]
        row.update(kin)
        attempt_rows.append(row)

    attempts_df = pd.DataFrame(attempt_rows)
    if attempts_df.empty:
        return pd.DataFrame(columns=["nfl_id"])

    if combine_results_df is not None and not combine_results_df.empty and "nfl_id" in combine_results_df.columns:
        res_cols = {
            "three_cone": "three_cone",
            "forty_yd": "forty",
            "shuttle": "short_shuttle",
        }
        for drill_name, col_name in res_cols.items():
            if col_name in combine_results_df.columns:
                bench = (
                    combine_results_df[["nfl_id", col_name]]
                    .dropna()
                    .drop_duplicates("nfl_id")
                    .rename(columns={col_name: "_bench_time"})
                )
                mask = attempts_df["drill"] == drill_name
                if mask.any() and not bench.empty:
                    merged_sub = attempts_df.loc[mask].merge(bench, on="nfl_id", how="left")
                    diff = (merged_sub["attempt_time_s"] - merged_sub["_bench_time"]).abs()
                    attempts_df.loc[mask, "attempt_time_s"] = diff.fillna(merged_sub["attempt_time_s"]).to_numpy()

    best_attempts = _pick_best_attempts(attempts_df)

    avail_metrics = [c for c in DRILL_METRIC_COLS if c in best_attempts.columns]
    pivoted = best_attempts.pivot(index="nfl_id", columns="drill", values=avail_metrics)
    pivoted.columns = [f"{drill}__{metric}" for metric, drill in pivoted.columns]
    out = pivoted.reset_index()

    if (
        drill_phases_df is not None
        and not drill_phases_df.empty
        and {"nfl_id", "attempt", "phase"}.issubset(drill_phases_df.columns)
    ):
        best_keys = best_attempts[["nfl_id", "attempt"]].drop_duplicates()
        best_phases = drill_phases_df.merge(best_keys, on=["nfl_id", "attempt"], how="inner")
        phase_metric_cols = [
            c
            for c in [
                "duration_s",
                "dist_yds",
                "speed_drop_yds",
                "speed_gain_yds",
                "exit_speed_yds",
                "peak_braking_yds2",
                "peak_accel_yds2",
                "max_turn_rate_deg_s",
                "total_turn_deg",
            ]
            if c in best_phases.columns
        ]
        if phase_metric_cols:
            p_piv = best_phases.pivot_table(
                index="nfl_id", columns="phase", values=phase_metric_cols, aggfunc="max"
            )
            p_piv.columns = [f"phase__{phase}__{metric}" for metric, phase in p_piv.columns]
            out = out.merge(p_piv.reset_index(), on="nfl_id", how="left")

    return out

In [47]:
ROUTE_KINEMATIC_COLS = [
    "duration_s",
    "dist_yds",
    "start_speed_yds",
    "exit_speed_yds",
    "min_speed_yds",
    "peak_speed_yds",
    "speed_drop_yds",
    "speed_gain_yds",
    "peak_braking_yds2",
    "mean_braking_yds2",
    "peak_accel_yds2",
    "max_turn_rate_deg_s",
    "total_turn_deg",
]


def load_route_tracking_from_sqlite(
    conn: sqlite3.Connection,
    tracking_table: str = "game_tracking",
    positions: tuple[str, ...] = ("WR", "TE"),
) -> pd.DataFrame:
    """
    Pulls receiver route tracking (`ball_snap` -> `pass_forward`) from the
    combined multi-season SQLite tracking table without join fan-out.
    """
    pos_sql = ", ".join(f"'{p}'" for p in positions)
    query = f"""
    WITH dedup_players AS (
        SELECT
            nfl_id,
            MAX(nfl_position) AS nfl_position
        FROM players
        GROUP BY nfl_id
    ),
    dedup_player_play AS (
        SELECT
            game_id,
            play_id,
            nfl_id,
            MAX(lined_up_position) AS lined_up_position,
            MAX(route_ran) AS route_ran,
            MAX(target) AS target,
            MAX(separation_at_pass_forward) AS separation_at_pass_forward,
            MAX(yards_after_catch) AS yards_after_catch,
            MAX(expected_yards_after_catch) AS expected_yards_after_catch,
            MAX(team_coverage_man_zone) AS team_coverage_man_zone
        FROM player_play
        GROUP BY game_id, play_id, nfl_id
    ),
    eligible_receivers AS (
        SELECT
            pp.game_id,
            pp.play_id,
            pp.nfl_id,
            pp.route_ran,
            pp.target,
            pp.separation_at_pass_forward,
            pp.yards_after_catch,
            pp.expected_yards_after_catch,
            pp.team_coverage_man_zone
        FROM dedup_player_play AS pp
        LEFT JOIN dedup_players AS p
            ON pp.nfl_id = p.nfl_id
        WHERE pp.route_ran IS NOT NULL
          AND TRIM(pp.route_ran) != ''
          AND (
              p.nfl_position IN ({pos_sql})
              OR pp.lined_up_position IN ({pos_sql})
          )
    ),
    play_windows AS (
        SELECT
            gt.game_id,
            gt.play_id,
            gt.nfl_id,
            MIN(CASE WHEN gt.event = 'ball_snap' THEN gt.time END) AS snap_time,
            MIN(CASE WHEN gt.event = 'pass_forward' THEN gt.time END) AS pass_fwd_time
        FROM {tracking_table} AS gt
        INNER JOIN eligible_receivers AS er
            ON gt.game_id = er.game_id
           AND gt.play_id = er.play_id
           AND gt.nfl_id = er.nfl_id
        WHERE gt.event IN ('ball_snap', 'pass_forward')
        GROUP BY gt.game_id, gt.play_id, gt.nfl_id
        HAVING snap_time IS NOT NULL
           AND pass_fwd_time IS NOT NULL
           AND snap_time <= pass_fwd_time
    )
    SELECT
        gt.game_id,
        gt.play_id,
        gt.nfl_id,
        gt.time,
        gt.x,
        gt.y,
        gt.s,
        gt.a,
        gt.dis,
        gt.o,
        gt.dir,
        gt.event,
        er.route_ran,
        er.target,
        er.separation_at_pass_forward,
        er.yards_after_catch,
        er.expected_yards_after_catch,
        er.team_coverage_man_zone
    FROM {tracking_table} AS gt
    INNER JOIN play_windows AS pw
        ON gt.game_id = pw.game_id
       AND gt.play_id = pw.play_id
       AND gt.nfl_id = pw.nfl_id
       AND gt.time BETWEEN pw.snap_time AND pw.pass_fwd_time
    INNER JOIN eligible_receivers AS er
        ON gt.game_id = er.game_id
       AND gt.play_id = er.play_id
       AND gt.nfl_id = er.nfl_id
    """
    df = pd.read_sql_query(query, conn)
    # Guard against exact duplicate rows in tracking
    return df.drop_duplicates(subset=["game_id", "play_id", "nfl_id", "time"]).reset_index(drop=True)


def build_route_tables_player_route(
    route_tracking_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns `(route_kinematics_player_route, route_outcomes_player_route)`,
    each with 1 row per `(nfl_id, route_ran)` and aligned kinematic column names.
    """
    if route_tracking_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = route_tracking_df.copy()
    df["_t_parsed"] = parse_tracking_time(df["time"])

    play_records = []
    for (game_id, play_id, nfl_id), g in df.groupby(["game_id", "play_id", "nfl_id"], dropna=False, sort=False):
        g = g.sort_values("_t_parsed").reset_index(drop=True)
        if len(g) < 3:
            continue

        kin = compute_segment_kinematics(g)
        meta = g.iloc[0]

        yac = meta.get("yards_after_catch", np.nan)
        xyac = meta.get("expected_yards_after_catch", np.nan)
        yac_oe = (float(yac) - float(xyac)) if (pd.notna(yac) and pd.notna(xyac)) else np.nan

        play_records.append({
            "game_id": game_id,
            "play_id": play_id,
            "nfl_id": nfl_id,
            "route_ran": meta.get("route_ran"),
            "target": meta.get("target", np.nan),
            "separation_at_pass_forward": meta.get("separation_at_pass_forward", np.nan),
            "yards_after_catch": yac,
            "expected_yards_after_catch": xyac,
            "yac_over_exp": yac_oe,
            **kin,
        })

    plays_df = pd.DataFrame(play_records)
    if plays_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    # 2. Route Kinematics: 1 row per (nfl_id, route_ran) using consistent metric names
    kin_agg_map = {
        "n_routes": ("play_id", "count"),
        "duration_s": ("duration_s", "median"),
        "dist_yds": ("dist_yds", "median"),
        "start_speed_yds": ("start_speed_yds", "median"),
        "exit_speed_yds": ("exit_speed_yds", "median"),
        "min_speed_yds": ("min_speed_yds", "median"),
        "peak_speed_yds": ("peak_speed_yds", lambda s: float(np.nanpercentile(s, 75)) if s.notna().any() else np.nan),
        "speed_drop_yds": ("speed_drop_yds", "median"),
        "speed_gain_yds": ("speed_gain_yds", "median"),
        "peak_braking_yds2": ("peak_braking_yds2", "median"),
        "mean_braking_yds2": ("mean_braking_yds2", "median"),
        "peak_accel_yds2": ("peak_accel_yds2", "median"),
        "max_turn_rate_deg_s": ("max_turn_rate_deg_s", "median"),
        "total_turn_deg": ("total_turn_deg", "median"),
    }
    valid_kin_agg = {k: v for k, v in kin_agg_map.items() if v[0] in plays_df.columns}
    route_kin_df = (
        plays_df.groupby(["nfl_id", "route_ran"], as_index=False)
        .agg(**valid_kin_agg)
        .sort_values(["nfl_id", "route_ran"])
        .reset_index(drop=True)
    )

    # 3. Route Outcomes: 1 row per (nfl_id, route_ran)
    out_agg_map = {
        "n_routes": ("play_id", "count"),
        "n_targets": ("target", lambda s: float(np.nansum(s)) if s.notna().any() else np.nan),
        "target_rate": ("target", lambda s: float(np.nanmean(s)) if s.notna().any() else np.nan),
        "sep_at_pass_forward_mean": ("separation_at_pass_forward", "mean"),
        "sep_at_pass_forward_med": ("separation_at_pass_forward", "median"),
        "yac_mean": ("yards_after_catch", "mean"),
        "xyac_mean": ("expected_yards_after_catch", "mean"),
        "yac_over_exp_mean": ("yac_over_exp", "mean"),
    }
    valid_out_agg = {k: v for k, v in out_agg_map.items() if v[0] in plays_df.columns}
    route_out_df = (
        plays_df.groupby(["nfl_id", "route_ran"], as_index=False)
        .agg(**valid_out_agg)
        .sort_values(["nfl_id", "route_ran"])
        .reset_index(drop=True)
    )

    return route_kin_df, route_out_df

In [48]:
with sqlite3.connect("../data/bdb2027.sqlite") as conn:
    route_tracking_df = load_route_tracking_from_sqlite(conn, tracking_table="game_tracking_all")
    combine_results_df = pd.read_sql_query("SELECT * FROM df_combine_results", conn)
    combine_tracking_df = pd.read_sql_query("SELECT * FROM df_combine_tracking", conn)

drill_kin_player = build_drill_kinematics_player(
    combine_tracking_df=combine_tracking_df,
    combine_results_df=combine_results_df,
)
route_kin_player_route, route_out_player_route = build_route_tables_player_route(route_tracking_df)

# Part 2: Model Construction

In [49]:
ROUTE_CLASSES: dict[str, set[str]] = {
    "SHARP": {"IN", "OUT", "HITCH", "FLAT"},
    "MODERATE": {"POST", "CORNER", "WHEEL", "SLANT", "CROSS"},
    "MINIMAL": {"GO", "SCREEN"},
}
ROUTE_CLASSES["RELEVANT"] = ROUTE_CLASSES["SHARP"] | ROUTE_CLASSES["MODERATE"]

BROAD_COVERAGES = {"MAN", "ZONE"}
SPECIFIC_COVERAGES = {
    "COVER_1",
    "COVER_2",
    "COVER_3",
    "COVER_4",
    "COVER_6_ZONE",
    "COVER_0",
    "PREVENT",
}

OUTCOME_COLS_PRIORITY = [
    "separation_at_pass_forward",
    "yards_after_catch",
    "expected_yards_after_catch",
    "pass_result",
    "target",
    "rec_yards",
]


def classify_route(route_ran: str | None) -> str:
    if not route_ran or pd.isna(route_ran):
        return "UNKNOWN"
    r = str(route_ran).strip().upper()
    if r in ROUTE_CLASSES["SHARP"]:
        return "SHARP"
    if r in ROUTE_CLASSES["MODERATE"]:
        return "MODERATE"
    if r in ROUTE_CLASSES["MINIMAL"]:
        return "MINIMAL"
    return "OTHER"


def _normalize_coverage_broad(val: str | None) -> str:
    if not val or pd.isna(val):
        return "UNKNOWN"
    s = str(val).strip().upper()
    if "MAN" in s:
        return "MAN"
    if "ZONE" in s or s.startswith("COVER_"):
        return "ZONE"
    return s


def _normalize_coverage_specific(val: str | None) -> str:
    if not val or pd.isna(val):
        return "UNKNOWN"
    return str(val).strip().upper().replace(" ", "_")


def filter_route_plays(
    plays_df: pd.DataFrame,
    route_name: str = "ALL",
    route_class: str = "ALL",
    coverage_broad: str = "ALL",
    coverage_specific: str = "ALL",
) -> pd.DataFrame:
    """
    Filters play-level route data by route name, route classification,
    broad coverage (`MAN`, `ZONE`), and specific coverage (`COVER_1`, etc.).
    """
    df = plays_df.copy()
    if df.empty:
        return df

    df["_route_upper"] = df["route_ran"].astype(str).str.strip().str.upper()
    df["_route_class"] = df["_route_upper"].map(classify_route)

    if route_name and route_name.upper() != "ALL":
        df = df[df["_route_upper"] == route_name.strip().upper()]

    rc = route_class.strip().upper() if route_class else "ALL"
    if rc != "ALL":
        if rc not in ROUTE_CLASSES:
            raise ValueError(f"Unsupported route_class '{route_class}'. Expected one of {list(ROUTE_CLASSES)} or 'ALL'.")
        df = df[df["_route_class"].isin(
            ROUTE_CLASSES[rc] if rc != "RELEVANT" else ROUTE_CLASSES["RELEVANT"]
        )]

    cov_col = "team_coverage_man_zone" if "team_coverage_man_zone" in df.columns else None
    spec_col = (
        "coverage_type"
        if "coverage_type" in df.columns
        else ("team_coverage_type" if "team_coverage_type" in df.columns else cov_col)
    )

    cb = coverage_broad.strip().upper() if coverage_broad else "ALL"
    if cb != "ALL" and cov_col is not None:
        df["_cov_broad"] = df[cov_col].map(_normalize_coverage_broad)
        df = df[df["_cov_broad"] == cb]

    cs = coverage_specific.strip().upper() if coverage_specific else "ALL"
    if cs != "ALL" and spec_col is not None:
        df["_cov_spec"] = df[spec_col].map(_normalize_coverage_specific)
        df = df[df["_cov_spec"] == cs]

    return df.drop(columns=[c for c in ["_route_upper", "_route_class", "_cov_broad", "_cov_spec"] if c in df.columns])

In [50]:
def build_play_level_route_table(route_tracking_df: pd.DataFrame) -> pd.DataFrame:
    """
    Builds 1 row per `(game_id, play_id, nfl_id)` with kinematics and outcome
    columns from `route_tracking_df`.
    """
    if route_tracking_df.empty:
        return pd.DataFrame()

    df = route_tracking_df.copy()
    df["_t_parsed"] = parse_tracking_time(df["time"])

    records = []
    for (game_id, play_id, nfl_id), g in df.groupby(["game_id", "play_id", "nfl_id"], dropna=False, sort=False):
        g = g.sort_values("_t_parsed").reset_index(drop=True)
        if len(g) < 3:
            continue

        kin = compute_segment_kinematics(g)
        meta = g.iloc[0]

        yac = meta.get("yards_after_catch", np.nan)
        xyac = meta.get("expected_yards_after_catch", np.nan)
        rec_yds = meta.get("rec_yards", meta.get("receiving_yards", yac))

        # Encode `pass_result` numerically if string labels exist (e.g., COMPLETE / INCOMPLETE / INTERCEPTION)
        raw_pr = meta.get("pass_result", np.nan)
        if isinstance(raw_pr, str):
            pr_map = {"C": 1.0, "COMPLETE": 1.0, "COMPLETION": 1.0, "I": 0.0, "INCOMPLETE": 0.0, "INT": -1.0, "INTERCEPTION": -1.0}
            pass_result_num = pr_map.get(raw_pr.strip().upper(), np.nan)
        else:
            pass_result_num = float(raw_pr) if pd.notna(raw_pr) else np.nan

        records.append({
            "game_id": game_id,
            "play_id": play_id,
            "nfl_id": nfl_id,
            "route_ran": meta.get("route_ran"),
            "team_coverage_man_zone": meta.get("team_coverage_man_zone"),
            "coverage_type": meta.get("coverage_type", meta.get("team_coverage_type")),
            "separation_at_pass_forward": meta.get("separation_at_pass_forward", np.nan),
            "yards_after_catch": yac,
            "expected_yards_after_catch": xyac,
            "pass_result": pass_result_num,
            "target": meta.get("target", np.nan),
            "rec_yards": rec_yds,
            **kin,
        })

    return pd.DataFrame(records)


def aggregate_player_route_hyperparams(
    plays_df: pd.DataFrame,
    route_name: str = "ALL",
    route_class: str = "ALL",
    coverage_broad: str = "ALL",
    coverage_specific: str = "ALL",
    min_plays_per_player: int = 1,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Filters plays by hyperparameters and aggregates to 1 row per `nfl_id` for:
      1. `player_route_kin` (kinematics prefixed with `route__`)
      2. `player_route_out` (outcomes prefixed with `out__`)
    """
    empty_kin = pd.DataFrame(columns=["nfl_id", "n_plays"])
    empty_out = pd.DataFrame(columns=["nfl_id", "n_plays"])

    if plays_df is None or plays_df.empty:
        return empty_kin, empty_out

    sub = filter_route_plays(
        plays_df,
        route_name=route_name,
        route_class=route_class,
        coverage_broad=coverage_broad,
        coverage_specific=coverage_specific,
    )
    if sub.empty:
        return empty_kin, empty_out

    kin_cols = [
        c
        for c in [
            "duration_s",
            "dist_yds",
            "start_speed_yds",
            "exit_speed_yds",
            "min_speed_yds",
            "peak_speed_yds",
            "speed_drop_yds",
            "speed_gain_yds",
            "peak_braking_yds2",
            "mean_braking_yds2",
            "peak_accel_yds2",
            "max_turn_rate_deg_s",
            "total_turn_deg",
        ]
        if c in sub.columns
    ]

    kin_agg: dict[str, tuple] = {"n_plays": ("play_id", "count")}
    for c in kin_cols:
        if c == "peak_speed_yds":
            kin_agg[f"route__{c}"] = (
                c,
                lambda s: float(np.nanpercentile(s.dropna(), 75)) if s.notna().any() else np.nan,
            )
        else:
            kin_agg[f"route__{c}"] = (c, "median")

    player_kin = sub.groupby("nfl_id", as_index=False).agg(**kin_agg)
    player_kin = player_kin[player_kin["n_plays"] >= min_plays_per_player].reset_index(drop=True)

    out_agg: dict[str, tuple] = {"n_plays": ("play_id", "count")}
    for c in OUTCOME_COLS_PRIORITY:
        if c in sub.columns:
            out_agg[f"out__{c}"] = (c, "mean")

    player_out = sub.groupby("nfl_id", as_index=False).agg(**out_agg)
    player_out = player_out[player_out["n_plays"] >= min_plays_per_player].reset_index(drop=True)

    if player_kin.empty:
        player_kin = empty_kin
    if player_out.empty:
        player_out = empty_out

    return player_kin, player_out

In [51]:
def _pairwise_corr_fdr(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_cols: list[str],
    right_cols: list[str],
    method: str = "spearman",
    min_players: int = 10,
    alpha: float = 0.05,
) -> pd.DataFrame:
    merged = pd.merge(left_df, right_df, on="nfl_id", how="inner")
    rows = []

    for l_col in left_cols:
        for r_col in right_cols:
            if l_col == r_col:
                continue
            pair = merged[[l_col, r_col]].dropna()
            n_obs = len(pair)
            if n_obs < min_players or pair[l_col].nunique() < 2 or pair[r_col].nunique() < 2:
                continue

            if method == "pearson":
                stat_val, p_val = stats.pearsonr(pair[l_col], pair[r_col])
            elif method == "spearman":
                stat_val, p_val = stats.spearmanr(pair[l_col], pair[r_col])
            else:
                raise ValueError("method must be 'pearson' or 'spearman'")

            if pd.notna(stat_val) and pd.notna(p_val):
                rows.append({
                    "left_metric": l_col,
                    "right_metric": r_col,
                    "n_players": n_obs,
                    "corr": float(stat_val),
                    "p_value": float(p_val),
                    "abs_corr": abs(float(stat_val)),
                })

    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows)
    reject_fdr, q_vals, _, _ = multipletests(out["p_value"].to_numpy(), alpha=alpha, method="fdr_bh")
    out["p_fdr_bh"] = q_vals
    out["sig_raw"] = out["p_value"] < alpha
    out["sig_fdr"] = reject_fdr
    return (
        out.sort_values(["sig_fdr", "abs_corr"], ascending=[False, False])
        .drop(columns=["abs_corr"])
        .reset_index(drop=True)
    )


def compare_drill_to_route_kinematics(
    drill_kin_player: pd.DataFrame,
    player_route_kin: pd.DataFrame,
    method: str = "spearman",
    min_players: int = 10,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Step 2.1: Correlates 1-row-per-player drill kinematics against
    1-row-per-player route kinematics with BH-FDR adjustment.
    """
    drill_cols = [
        c for c in drill_kin_player.select_dtypes(include=[np.number]).columns if c != "nfl_id"
    ]
    route_cols = [
        c
        for c in player_route_kin.select_dtypes(include=[np.number]).columns
        if c not in {"nfl_id", "n_plays"}
    ]
    res = _pairwise_corr_fdr(
        drill_kin_player,
        player_route_kin,
        left_cols=drill_cols,
        right_cols=route_cols,
        method=method,
        min_players=min_players,
        alpha=alpha,
    )
    if not res.empty:
        res = res.rename(columns={"left_metric": "drill_metric", "right_metric": "route_metric"})
    return res

In [52]:
def compare_route_kin_to_outcomes_by_player(
    plays_df: pd.DataFrame,
    route_name: str = "ALL",
    route_class: str = "ALL",
    coverage_broad: str = "ALL",
    coverage_specific: str = "ALL",
    min_plays_per_player: int = 3,
    min_players: int = 10,
    method: str = "spearman",
    alpha: float = 0.05,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Step 2.2: Aggregates route kinematics and outcomes per `nfl_id` under the
    requested hyperparameters, then correlates kinematics against outcomes across players.
    Returns `(corr_table, merged_player_df)`.
    """
    player_kin, player_out = aggregate_player_route_hyperparams(
        plays_df,
        route_name=route_name,
        route_class=route_class,
        coverage_broad=coverage_broad,
        coverage_specific=coverage_specific,
        min_plays_per_player=min_plays_per_player,
    )
    if player_kin.empty or player_out.empty:
        return pd.DataFrame(), pd.DataFrame()

    kin_cols = [
        c for c in player_kin.select_dtypes(include=[np.number]).columns if c not in {"nfl_id", "n_plays"}
    ]
    out_cols = [
        c for c in player_out.select_dtypes(include=[np.number]).columns if c not in {"nfl_id", "n_plays"}
    ]

    corr_df = _pairwise_corr_fdr(
        player_kin,
        player_out,
        left_cols=kin_cols,
        right_cols=out_cols,
        method=method,
        min_players=min_players,
        alpha=alpha,
    )
    if not corr_df.empty:
        corr_df = corr_df.rename(columns={"left_metric": "route_metric", "right_metric": "outcome_metric"})
        corr_df.insert(0, "coverage_specific", coverage_specific)
        corr_df.insert(0, "coverage_broad", coverage_broad)
        corr_df.insert(0, "route_class", route_class)
        corr_df.insert(0, "route_name", route_name)

    merged_player = pd.merge(
        player_kin,
        player_out.drop(columns=[c for c in ["n_plays"] if c in player_out.columns]),
        on="nfl_id",
        how="inner",
    )
    return corr_df, merged_player

In [53]:
FOCUS_OUTCOMES = [
    "out__separation_at_pass_forward",
    "out__yards_after_catch",
    "out__expected_yards_after_catch",
    "out__pass_result",
    "out__target",
    "out__rec_yards",
]


def scan_hyperparameter_grid(
    plays_df: pd.DataFrame,
    drill_kin_player: pd.DataFrame,
    route_names: list[str] = ("ALL", "OUT", "SLANT", "CROSS", "POST", "CORNER", "HITCH", "FLAT", "WHEEL", "GO", "SCREEN"),
    route_classes: list[str] = ("ALL", "SHARP", "MODERATE", "MINIMAL", "RELEVANT"),
    coverage_broads: list[str] = ("ALL", "MAN", "ZONE"),
    coverage_specifics: list[str] = ("ALL", "COVER_1", "COVER_2", "COVER_3", "COVER_4", "COVER_6_ZONE", "COVER_0", "PREVENT"),
    min_plays_per_player: int = 3,
    min_players: int = 10,
    method: str = "spearman",
    alpha: float = 0.05,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs Step 2.1 and Step 2.2 across hyperparameter combinations and returns:
      - `step2_1_df`: drill-to-route kinematic correlations
      - `step2_2_df`: player-level route-kinematics-to-outcomes correlations
    """
    # Step 2.1 across route classes & broad coverages
    s21_frames = []
    for rc in route_classes:
        for cb in coverage_broads:
            p_kin, _ = aggregate_player_route_hyperparams(
                plays_df,
                route_name="ALL",
                route_class=rc,
                coverage_broad=cb,
                coverage_specific="ALL",
                min_plays_per_player=min_plays_per_player,
            )
            if p_kin.empty:
                continue
            df_21 = compare_drill_to_route_kinematics(
                drill_kin_player, p_kin, method=method, min_players=min_players, alpha=alpha
            )
            if not df_21.empty:
                df_21["route_class"] = rc
                df_21["coverage_broad"] = cb
                s21_frames.append(df_21)

    step2_1_df = pd.concat(s21_frames, ignore_index=True) if s21_frames else pd.DataFrame()

    # Step 2.2 across route_names, route_classes, and coverages
    s22_frames = []
    for rn in route_names:
        for rc in route_classes:
            for cb in coverage_broads:
                for cs in coverage_specifics:
                    df_22, _ = compare_route_kin_to_outcomes_by_player(
                        plays_df,
                        route_name=rn,
                        route_class=rc,
                        coverage_broad=cb,
                        coverage_specific=cs,
                        min_plays_per_player=min_plays_per_player,
                        min_players=min_players,
                        method=method,
                        alpha=alpha,
                    )
                    if not df_22.empty:
                        s22_frames.append(df_22)

    step2_2_df = pd.concat(s22_frames, ignore_index=True) if s22_frames else pd.DataFrame()
    return step2_1_df, step2_2_df


def correlate_surviving_correlations_to_outcomes(
    plays_df: pd.DataFrame,
    step2_1_df: pd.DataFrame,
    step2_2_df: pd.DataFrame,
    use_fdr_survivors: bool = True,
    min_plays_per_player: int = 3,
    method: str = "spearman",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Step 2.3:
      1. Filters to `route_metric` drivers that survive both Step 2.1 (drill <-> route kin)
         and Step 2.2 (route kin <-> route outcomes), then tests whether higher values of
         those surviving drivers yield higher player outcomes (`separation_at_pass_forward`,
         `yards_after_catch`, `expected_yards_after_catch`, `pass_result`, `target`, `rec_yards`).
      2. Correlates surviving correlation strengths (`abs_corr` / `corr`) across slices against
         slice-level outcome levels to see if stronger correlations yield better outcomes.
    """
    if step2_1_df.empty or step2_2_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    sig_col = "sig_fdr" if (use_fdr_survivors and "sig_fdr" in step2_1_df.columns) else "sig_raw"
    surv_21 = step2_1_df[step2_1_df[sig_col]] if sig_col in step2_1_df.columns else step2_1_df
    surv_22 = step2_2_df[step2_2_df[sig_col]] if sig_col in step2_2_df.columns else step2_2_df

    surviving_route_metrics = sorted(
        set(surv_21["route_metric"].dropna()).intersection(set(surv_22["route_metric"].dropna()))
    )

    driver_rows = []
    slice_meta_rows = []

    slice_keys = ["route_name", "route_class", "coverage_broad", "coverage_specific"]
    avail_keys = [k for k in slice_keys if k in step2_2_df.columns]

    for keys, grp in step2_2_df.groupby(avail_keys, dropna=False, sort=False):
        k_dict = dict(zip(avail_keys, keys if isinstance(keys, tuple) else (keys,)))
        p_kin, p_out = aggregate_player_route_hyperparams(
            plays_df,
            route_name=k_dict.get("route_name", "ALL"),
            route_class=k_dict.get("route_class", "ALL"),
            coverage_broad=k_dict.get("coverage_broad", "ALL"),
            coverage_specific=k_dict.get("coverage_specific", "ALL"),
            min_plays_per_player=min_plays_per_player,
        )
        if p_kin.empty or p_out.empty:
            continue

        merged = pd.merge(
            p_kin,
            p_out.drop(columns=[c for c in ["n_plays"] if c in p_out.columns]),
            on="nfl_id",
            how="inner",
        )
        avail_outcomes = [c for c in FOCUS_OUTCOMES if c in merged.columns]

        # Slice-level summary of outcomes and surviving correlation strengths
        grp_surv = grp[grp[sig_col]] if sig_col in grp.columns else grp
        max_surv_abs = float(grp_surv["abs_corr"].max()) if not grp_surv.empty and "abs_corr" in grp_surv.columns else np.nan
        mean_surv_abs = float(grp_surv["abs_corr"].mean()) if not grp_surv.empty and "abs_corr" in grp_surv.columns else np.nan
        n_surv_pairs = int(len(grp_surv))

        slice_record = {
            **k_dict,
            "n_players": len(merged),
            "n_surviving_pairs": n_surv_pairs,
            "max_surv_abs_corr": max_surv_abs,
            "mean_surv_abs_corr": mean_surv_abs,
        }
        for oc in avail_outcomes:
            slice_record[f"slice_mean_{oc}"] = float(merged[oc].mean()) if merged[oc].notna().any() else np.nan
        slice_meta_rows.append(slice_record)

        # Test each surviving route_metric driver in this slice against outcomes
        for m_col in surviving_route_metrics:
            if m_col not in merged.columns:
                continue
            m_series = merged[m_col]
            if m_series.notna().sum() < 8 or m_series.nunique() < 3:
                continue

            q75 = m_series.quantile(0.75)
            q25 = m_series.quantile(0.25)
            top_q = merged[m_series >= q75]
            bot_q = merged[m_series <= q25]

            for oc in avail_outcomes:
                pair = merged[[m_col, oc]].dropna()
                if len(pair) < 8 or pair[oc].nunique() < 2:
                    continue
                r_val, p_val = (
                    stats.pearsonr(pair[m_col], pair[oc])
                    if method == "pearson"
                    else stats.spearmanr(pair[m_col], pair[oc])
                )
                top_mean = float(top_q[oc].mean()) if top_q[oc].notna().any() else np.nan
                bot_mean = float(bot_q[oc].mean()) if bot_q[oc].notna().any() else np.nan

                driver_rows.append({
                    **k_dict,
                    "route_metric": m_col,
                    "outcome_metric": oc.replace("out__", ""),
                    "is_priority_3": oc.replace("out__", "") in {
                        "separation_at_pass_forward",
                        "yards_after_catch",
                        "expected_yards_after_catch",
                    },
                    "n_players": len(pair),
                    "driver_to_outcome_corr": float(r_val) if pd.notna(r_val) else np.nan,
                    "driver_to_outcome_p": float(p_val) if pd.notna(p_val) else np.nan,
                    "top_quartile_outcome_mean": top_mean,
                    "bottom_quartile_outcome_mean": bot_mean,
                    "top_minus_bottom_lift": (
                        (top_mean - bot_mean) if (pd.notna(top_mean) and pd.notna(bot_mean)) else np.nan
                    ),
                })

    driver_df = pd.DataFrame(driver_rows)
    if not driver_df.empty:
        driver_df = driver_df.sort_values(
            ["is_priority_3", "top_minus_bottom_lift"], ascending=[False, False], key=lambda s: s.abs() if s.dtype.kind in "fc" else s
        ).reset_index(drop=True)

    slice_df = pd.DataFrame(slice_meta_rows)
    # Correlate correlation strength across slices against slice outcome levels
    meta_corr_rows = []
    if not slice_df.empty:
        for strength_col in ["max_surv_abs_corr", "mean_surv_abs_corr", "n_surviving_pairs"]:
            for oc in [c for c in FOCUS_OUTCOMES if f"slice_mean_{c}" in slice_df.columns]:
                s_col = f"slice_mean_{oc}"
                pair = slice_df[[strength_col, s_col]].dropna()
                if len(pair) >= 5 and pair[strength_col].nunique() >= 2 and pair[s_col].nunique() >= 2:
                    r_val, p_val = (
                        stats.pearsonr(pair[strength_col], pair[s_col])
                        if method == "pearson"
                        else stats.spearmanr(pair[strength_col], pair[s_col])
                    )
                    meta_corr_rows.append({
                        "correlation_strength_metric": strength_col,
                        "outcome_metric": oc.replace("out__", ""),
                        "is_priority_3": oc.replace("out__", "") in {
                            "separation_at_pass_forward",
                            "yards_after_catch",
                            "expected_yards_after_catch",
                        },
                        "n_slices": len(pair),
                        "corr_of_corr_to_outcome": float(r_val) if pd.notna(r_val) else np.nan,
                        "p_value": float(p_val) if pd.notna(p_val) else np.nan,
                    })

    meta_corr_df = pd.DataFrame(meta_corr_rows)
    if not meta_corr_df.empty:
        meta_corr_df = meta_corr_df.sort_values(
            ["is_priority_3", "corr_of_corr_to_outcome"], ascending=[False, False]
        ).reset_index(drop=True)

    # Attach meta-correlation summary or return clean tables; let's return driver_df and slice_df with meta summary
    if not meta_corr_df.empty:
        driver_df.attrs["meta_correlation_of_correlations"] = meta_corr_df
        slice_df.attrs["meta_correlation_of_correlations"] = meta_corr_df

    return driver_df, slice_df

## Sample Runthrough

In [100]:
plays_df = build_play_level_route_table(route_tracking_df)

# Step 2.1: Drill kinematics vs. route kinematics
player_kin_df, player_out_df = aggregate_player_route_hyperparams(
    plays_df,
    route_name="ALL",
    route_class="RELEVANT",
    coverage_broad="ALL",
    coverage_specific="ALL",
)
drill_vs_route_kin = compare_drill_to_route_kinematics(drill_kin_player, player_kin_df)

# Step 2.2: Player-by-player route kinematics vs. outcomes for a specific hyperparameter combo
route_vs_out_corr, player_merged_df = compare_route_kin_to_outcomes_by_player(
    plays_df,
    route_name="ALL",
    route_class="RELEVANT",
    coverage_broad="ALL",
    coverage_specific="ALL",
)

# Step 2.3: Scan across hyperparameters and correlate surviving correlations to outcomes
s21_all, s22_all = scan_hyperparameter_grid(plays_df, drill_kin_player)
surviving_driver_outcomes, slice_outcomes = correlate_surviving_correlations_to_outcomes(
    plays_df, s21_all, s22_all, use_fdr_survivors=True
)
meta_corr_results = surviving_driver_outcomes.attrs.get("meta_correlation_of_correlations")

In [101]:
surviving_driver_outcomes

,route_name,route_class,coverage_broad,coverage_specific,route_metric,outcome_metric,is_priority_3,n_players,driver_to_outcome_corr,driver_to_outcome_p,top_quartile_outcome_mean,bottom_quartile_outcome_mean,top_minus_bottom_lift
0,CORNER,ALL,MAN,ALL,route__speed_drop_yds,yards_after_catch,True,38,0.416130,0.009362,11.000000,1.000000,10.000000
1,CORNER,ALL,MAN,ALL,route__dist_yds,yards_after_catch,True,38,0.510383,0.001061,9.142857,0.000000,9.142857
2,POST,ALL,MAN,ALL,route__dist_yds,yards_after_catch,True,31,0.286597,0.118023,9.700000,1.272727,8.427273
3,CORNER,ALL,MAN,ALL,route__peak_accel_yds2,yards_after_catch,True,38,0.109317,0.513544,9.142857,3.642857,5.500000
4,CORNER,ALL,MAN,ALL,route__min_speed_yds,yards_after_catch,True,38,-0.275035,0.094683,1.700000,7.181818,-5.481818
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1345,CORNER,ALL,ALL,ALL,route__min_speed_yds,target,False,120,0.001445,0.987506,0.105638,0.104754,0.000884
1346,CORNER,ALL,ALL,ALL,route__start_speed_yds,target,False,120,0.001550,0.986599,0.105638,0.104754,0.000884
1347,ALL,ALL,ZONE,ALL,route__speed_drop_yds,target,False,143,-0.096184,0.253141,0.180272,0.179665,0.000607
1348,SLANT,ALL,MAN,ALL,route__speed_drop_yds,target,False,91,0.011628,0.912889,0.303396,0.303728,-0.000332


# Part 3: Does it mean anything?

In [75]:
def _metric_family(col_name: str) -> str:
    name = str(col_name).lower()
    if "brake" in name or "drop" in name or "decel" in name:
        return "braking_decel"
    if "burst" in name or "accel" in name or "gain" in name:
        return "burst_accel"
    if "turn" in name or "deg" in name:
        return "turn_angle"
    if "speed" in name or "forty" in name or "split" in name:
        return "speed_time"
    if "sep" in name:
        return "separation"
    if "yac" in name or "rec_yds" in name:
        return "yac_yards"
    if "target" in name or "pass_result" in name:
        return "target_result"
    return "other"


def summarize_broad_patterns(
    step2_1_df: pd.DataFrame,
    step2_2_df: pd.DataFrame,
    top_n: int = 15,
) -> dict[str, pd.DataFrame]:
    """
    Step 3.1: Extracts broad patterns across route classes, coverages,
    and kinematic/outcome families from Step 2.1 and Step 2.2 results.
    """
    outputs: dict[str, pd.DataFrame] = {}

    if not step2_1_df.empty:
        d21 = step2_1_df.copy()
        d21["drill_family"] = d21["drill_metric"].map(_metric_family)
        d21["route_family"] = d21["route_metric"].map(_metric_family)
        d21["abs_corr"] = d21["corr"].abs()

        grp_cols_21 = [c for c in ["route_class", "coverage_broad", "drill_family", "route_family"] if c in d21.columns]
        outputs["broad_drill_to_route"] = (
            d21.groupby(grp_cols_21, dropna=False, as_index=False)
            .agg(
                n_pairs=("corr", "count"),
                n_sig_fdr=("sig_fdr", "sum") if "sig_fdr" in d21.columns else ("corr", "count"),
                mean_corr=("corr", "mean"),
                median_abs_corr=("abs_corr", "median"),
                max_abs_corr=("abs_corr", "max"),
            )
            .sort_values(["n_sig_fdr", "median_abs_corr"], ascending=[False, False])
            .reset_index(drop=True)
        )
        outputs["top_drill_to_route_pairs"] = (
            d21.sort_values(["sig_fdr", "abs_corr"] if "sig_fdr" in d21.columns else ["abs_corr"], ascending=[False, False])
            .drop(columns=["abs_corr"])
            .head(top_n)
            .reset_index(drop=True)
        )

    if not step2_2_df.empty:
        d22 = step2_2_df.copy()
        d22["route_family"] = d22["route_metric"].map(_metric_family)
        d22["outcome_family"] = d22["outcome_metric"].map(_metric_family)
        d22["abs_corr"] = d22["corr"].abs()

        grp_cols_22 = [c for c in ["route_class", "coverage_broad", "route_family", "outcome_metric"] if c in d22.columns]
        outputs["broad_route_to_outcome"] = (
            d22.groupby(grp_cols_22, dropna=False, as_index=False)
            .agg(
                n_pairs=("corr", "count"),
                n_sig_fdr=("sig_fdr", "sum") if "sig_fdr" in d22.columns else ("corr", "count"),
                mean_corr=("corr", "mean"),
                median_abs_corr=("abs_corr", "median"),
                max_abs_corr=("abs_corr", "max"),
            )
            .sort_values(["n_sig_fdr", "median_abs_corr"], ascending=[False, False])
            .reset_index(drop=True)
        )
        outputs["top_route_to_outcome_pairs"] = (
            d22.sort_values(["sig_fdr", "abs_corr"] if "sig_fdr" in d22.columns else ["abs_corr"], ascending=[False, False])
            .drop(columns=["abs_corr"])
            .head(top_n)
            .reset_index(drop=True)
        )

    return outputs

In [76]:
def analyze_specific_wr_drill_performance(
    plays_df: pd.DataFrame,
    drill_kin_player: pd.DataFrame,
    players_df: pd.DataFrame,
    drill_metric_cols: list[str],
    routes_to_check: list[str] = ("ALL", "OUT", "SLANT", "CROSS", "POST", "CORNER", "HITCH", "FLAT", "WHEEL", "GO", "SCREEN"),
    min_plays_per_route: int = 3,
    min_wrs: int = 8,
    method: str = "spearman",
    alpha: float = 0.05,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Step 3.2: Filters to WRs (`nfl_position == 'WR'`), correlates specific drill metrics
    against general (`ALL`) and route-specific WR kinematics & outcomes, and builds
    a per-WR comparison table across routes.
    """
    wr_ids = (
        players_df[players_df["nfl_position"].astype(str).str.strip().str.upper() == "WR"]["nfl_id"]
        .dropna()
        .unique()
    )
    wr_drill = drill_kin_player[drill_kin_player["nfl_id"].isin(wr_ids)].copy()
    wr_plays = plays_df[plays_df["nfl_id"].isin(wr_ids)].copy()

    if wr_drill.empty or wr_plays.empty:
        return pd.DataFrame(), pd.DataFrame()

    avail_drill_cols = [c for c in drill_metric_cols if c in wr_drill.columns]
    if not avail_drill_cols:
        avail_drill_cols = [c for c in wr_drill.select_dtypes(include=[np.number]).columns if c != "nfl_id"]

    corr_rows = []
    wr_route_snapshots = []

    for r_name in routes_to_check:
        p_kin, p_out = aggregate_player_route_hyperparams(
            wr_plays,
            route_name=r_name,
            route_class="ALL",
            coverage_broad="ALL",
            coverage_specific="ALL",
            min_plays_per_player=min_plays_per_route,
        )
        if p_kin.empty or p_out.empty:
            continue

        merged = (
            wr_drill[["nfl_id"] + avail_drill_cols]
            .merge(p_kin, on="nfl_id", how="inner")
            .merge(p_out.drop(columns=[c for c in ["n_plays"] if c in p_out.columns]), on="nfl_id", how="inner")
        )
        if len(merged) < min_wrs:
            continue

        # Attach readable name if present in players_df
        name_col = next((c for c in ["player_name", "display_name", "full_name"] if c in players_df.columns), None)
        snap = merged.copy()
        snap["route_name"] = r_name
        if name_col:
            snap = snap.merge(players_df[["nfl_id", name_col]].drop_duplicates("nfl_id"), on="nfl_id", how="left")
        wr_route_snapshots.append(snap)

        target_cols = [
            c for c in merged.columns
            if c not in (["nfl_id", "n_plays"] + avail_drill_cols) and pd.api.types.is_numeric_dtype(merged[c])
        ]

        for d_col in avail_drill_cols:
            for t_col in target_cols:
                pair = merged[[d_col, t_col]].dropna()
                if len(pair) < min_wrs or pair[d_col].nunique() < 2 or pair[t_col].nunique() < 2:
                    continue
                stat_val, p_val = (
                    stats.pearsonr(pair[d_col], pair[t_col])
                    if method == "pearson"
                    else stats.spearmanr(pair[d_col], pair[t_col])
                )
                if pd.notna(stat_val) and pd.notna(p_val):
                    corr_rows.append({
                        "route_name": r_name,
                        "drill_metric": d_col,
                        "wr_route_metric": t_col,
                        "n_wrs": len(pair),
                        "corr": float(stat_val),
                        "p_value": float(p_val),
                        "abs_corr": abs(float(stat_val)),
                    })

    corr_df = pd.DataFrame(corr_rows)
    if not corr_df.empty:
        reject_fdr, q_vals, _, _ = multipletests(corr_df["p_value"].to_numpy(), alpha=alpha, method="fdr_bh")
        corr_df["p_fdr_bh"] = q_vals
        corr_df["sig_fdr"] = reject_fdr
        corr_df = (
            corr_df.sort_values(["sig_fdr", "abs_corr"], ascending=[False, False])
            .drop(columns=["abs_corr"])
            .reset_index(drop=True)
        )

    wr_profile_df = pd.concat(wr_route_snapshots, ignore_index=True) if wr_route_snapshots else pd.DataFrame()
    return corr_df, wr_profile_df

In [84]:
def run_robustness_checks(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    metric_pairs: list[tuple[str, str]],
    combine_results_df: pd.DataFrame | None = None,
    method: str = "spearman",
    n_boot: int = 1000,
    min_players: int = 8,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Step 3.3: Evaluates correlation stability via bootstrap 95% CI,
    leave-one-out max shift, and weight-residualized correlation.
    """
    rng = np.random.default_rng(random_state)

    l_df = left_df.copy()
    r_df = right_df.copy()
    l_df["nfl_id"] = pd.to_numeric(l_df["nfl_id"], errors="coerce")
    r_df["nfl_id"] = pd.to_numeric(r_df["nfl_id"], errors="coerce")

    # Drop overlapping non-key columns from right_df so pandas doesn't add _x/_y suffixes
    dup_cols = [c for c in r_df.columns if c in l_df.columns and c != "nfl_id"]
    if dup_cols:
        r_df = r_df.drop(columns=dup_cols)

    merged = pd.merge(l_df, r_df, on="nfl_id", how="inner")

    if combine_results_df is not None and "combine_weight" in combine_results_df.columns:
        w_df = combine_results_df[["nfl_id", "combine_weight"]].copy()
        w_df["nfl_id"] = pd.to_numeric(w_df["nfl_id"], errors="coerce")
        w_df = w_df.dropna(subset=["nfl_id"]).drop_duplicates("nfl_id")
        merged = merged.merge(w_df, on="nfl_id", how="left")

    def _resolve_col(col: str, cols: pd.Index) -> str | None:
        if col in cols:
            return col
        if col.startswith("out__out__") and col[5:] in cols:
            return col[5:]
        if f"out__{col}" in cols:
            return f"out__{col}"
        if f"route__{col}" in cols:
            return f"route__{col}"
        return None

    def _corr_fn(x: np.ndarray, y: np.ndarray) -> float:
        if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        if method == "pearson":
            return float(stats.pearsonr(x, y).statistic)
        return float(stats.spearmanr(x, y).statistic)

    def _residualize_on_weight(s: pd.Series, w: pd.Series) -> pd.Series:
        valid = pd.DataFrame({"s": s, "w": w}).dropna()
        if len(valid) < 5 or valid["w"].std() == 0:
            return s
        slope, intercept = np.polyfit(valid["w"], valid["s"], deg=1)
        return s - (intercept + slope * w)

    records = []
    for raw_l, raw_r in metric_pairs:
        l_col = _resolve_col(raw_l, merged.columns)
        r_col = _resolve_col(raw_r, merged.columns)
        if l_col is None or r_col is None or l_col == r_col:
            continue

        cols_needed = [l_col, r_col] + (["combine_weight"] if "combine_weight" in merged.columns else [])
        sub = merged[cols_needed].dropna(subset=[l_col, r_col]).reset_index(drop=True)
        n = len(sub)
        if n < min_players:
            continue

        x = sub[l_col].to_numpy(dtype=float)
        y = sub[r_col].to_numpy(dtype=float)
        base_r = _corr_fn(x, y)
        if np.isnan(base_r):
            continue

        boot_vals = np.empty(n_boot, dtype=float)
        for b in range(n_boot):
            idx = rng.integers(0, n, size=n)
            boot_vals[b] = _corr_fn(x[idx], y[idx])
        boot_vals = boot_vals[np.isfinite(boot_vals)]
        ci_low = float(np.percentile(boot_vals, 2.5)) if len(boot_vals) > 0 else np.nan
        ci_high = float(np.percentile(boot_vals, 97.5)) if len(boot_vals) > 0 else np.nan
        ci_excludes_zero = bool(np.isfinite(ci_low) and np.isfinite(ci_high) and ((ci_low > 0) or (ci_high < 0)))

        loo_shifts = []
        for i in range(n):
            mask = np.arange(n) != i
            r_i = _corr_fn(x[mask], y[mask])
            if np.isfinite(r_i):
                loo_shifts.append(abs(r_i - base_r))
        max_loo_shift = float(np.max(loo_shifts)) if loo_shifts else np.nan

        wt_corr = np.nan
        if "combine_weight" in sub.columns and sub["combine_weight"].notna().sum() >= min_players:
            w_sub = sub.dropna(subset=["combine_weight"])
            x_res = _residualize_on_weight(w_sub[l_col], w_sub["combine_weight"]).to_numpy(dtype=float)
            y_res = _residualize_on_weight(w_sub[r_col], w_sub["combine_weight"]).to_numpy(dtype=float)
            wt_corr = _corr_fn(x_res, y_res)

        records.append({
            "left_metric": l_col,
            "right_metric": r_col,
            "n_players": n,
            "corr": base_r,
            "boot_ci_low": ci_low,
            "boot_ci_high": ci_high,
            "ci_excludes_zero": ci_excludes_zero,
            "max_loo_shift": max_loo_shift,
            "weight_adj_corr": wt_corr,
            "robust_flag": bool(ci_excludes_zero and (pd.isna(max_loo_shift) or max_loo_shift < 0.12)),
        })

    if not records:
        return pd.DataFrame()

    return (
        pd.DataFrame(records)
        .sort_values(["robust_flag", "corr"], ascending=[False, False], key=lambda s: s.abs() if s.dtype.kind in "fc" else s)
        .reset_index(drop=True)
    )

In [85]:
# 3.1 Broad patterns across Step 2.1 and Step 2.2
broad_summary = summarize_broad_patterns(s21_all, s22_all, top_n=15)

# 3.2 Specific WR drill-to-route & outcome correlations + WR route profiles
wr_corr_df, wr_profile_df = analyze_specific_wr_drill_performance(
    plays_df=plays_df,
    drill_kin_player=drill_kin_player,
    players_df=pd.read_sql("SELECT * FROM df_players", con),
    drill_metric_cols=[
        "three_cone__attempt_time_s",
        "three_cone__peak_braking_yds2",
        "three_cone__max_turn_rate_deg_s",
        "shuttle__peak_braking_yds2",
        "shuttle__peak_accel_yds2",
        "forty_yd__peak_speed_yds",
    ],
)

# 3.3 Robustness check on top candidate pairs
top_pairs = list(
    zip(
        broad_summary["top_route_to_outcome_pairs"]["route_metric"].head(10),
        broad_summary["top_route_to_outcome_pairs"]["outcome_metric"].map(lambda c: f"out__{c}").head(10),
    )
)
robustness_df = run_robustness_checks(
    left_df=player_kin_df,
    right_df=player_out_df,
    metric_pairs=top_pairs,
    combine_results_df=combine_results_df,
)

In [98]:
robustness_df

,left_metric,right_metric,n_players,corr,boot_ci_low,boot_ci_high,ci_excludes_zero,max_loo_shift,weight_adj_corr,robust_flag
0,route__speed_drop_yds,out__expected_yards_after_catch,146,-0.217464,-0.377332,-0.054412,True,0.024628,-0.208062,True
1,route__speed_gain_yds,out__separation_at_pass_forward,153,-0.193406,-0.361063,-0.023146,True,0.023488,-0.133980,True
2,route__duration_s,out__yards_after_catch,143,0.116597,-0.054854,0.277792,False,0.019348,0.130636,False
3,route__duration_s,out__rec_yards,143,0.116597,-0.056999,0.283412,False,0.019348,0.130636,False
4,route__duration_s,out__expected_yards_after_catch,146,0.108147,-0.066321,0.259347,False,0.018787,0.123538,False
5,route__dist_yds,out__yards_after_catch,143,-0.065058,-0.230997,0.108543,False,0.020625,0.021293,False
6,route__dist_yds,out__rec_yards,143,-0.065058,-0.239282,0.109003,False,0.020625,0.021293,False
7,route__speed_drop_yds,out__target,153,0.038741,-0.142259,0.212694,False,0.020112,-0.173583,False
8,route__dist_yds,out__target,153,0.030963,-0.134733,0.189914,False,0.019966,-0.014790,False
9,route__peak_accel_yds2,out__separation_at_pass_forward,153,-0.015309,-0.194901,0.159501,False,0.020155,-0.000613,False
